# Sesión 02 — Inferencia Bayesiana
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo I · Fundamentos del Aprendizaje Estadístico**

## Objetivos de aprendizaje

Al finalizar esta sesión serás capaz de:

1. Identificar prior, verosimilitud y posterior en un problema clínico concreto.
2. Aplicar pares conjugados (Beta-Binomial, Gaussiana-Gaussiana) y visualizar la actualización secuencial.
3. Conectar la estimación MAP con la regularización L2.
4. Calcular VPP y VPN a partir de sensibilidad, especificidad y prevalencia, y explicar la falacia de la tasa base.
5. Distinguir cuándo usar EMV vs MAP vs inferencia bayesiana completa.

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Bishop (2006). *PRML*. §1.2 (Teoría de la probabilidad), §2.1–2.3 (distribuciones conjugadas). |
| ★★★ | Murphy (2022). *PML: An Introduction*. Cap. 4 (Estadística bayesiana). probml.ai |
| ★★☆ | Gelman et al. (2013). *Bayesian Data Analysis* (3ª ed.). Cap. 1–2. |
| ★★☆ | Altman, D.G. & Bland, J.M. (1994). Diagnostic tests 2: Predictive values. *BMJ* 309:102. |
| ★☆☆ | Casscells, W. et al. (1978). Interpretation by physicians of clinical laboratory results. *NEJM* 299:999–1001. (El experimento original de la falacia de la tasa base) |

## Parte 0 — Configuración

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.special import betaln

rng = np.random.default_rng(42)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
print('Configuración completa.')

## Parte 1 — El teorema de Bayes: prior, verosimilitud y posterior

El teorema de Bayes es la regla del producto aplicada dos veces:

$$\underbrace{p(\theta \mid \mathcal{D})}_{\text{posterior}} = \frac{\overbrace{p(\mathcal{D} \mid \theta)}^{\text{verosimilitud}} \; \overbrace{p(\theta)}^{\text{prior}}}{\underbrace{p(\mathcal{D})}_{\text{evidencia}}}$$

Donde $p(\mathcal{D}) = \int p(\mathcal{D}\mid\theta)\,p(\theta)\,d\theta$ es la constante de normalización.

### 1.1 Terminología clínica

| Término probabilístico | Significado clínico en diagnóstico |
|---|---|
| Prior $p(\theta)$ | Probabilidad pre-prueba (prevalencia en la población) |
| Verosimilitud $p(\mathcal{D}\mid\theta)$ | Probabilidad del resultado de la prueba dado el estado de salud |
| Posterior $p(\theta\mid\mathcal{D})$ | Probabilidad post-prueba (diagnóstico actualizado) |
| Evidencia $p(\mathcal{D})$ | Probabilidad marginal del resultado de la prueba |

In [ ]:
# ── Ejemplo introductorio: prueba de FA con ECG de una derivación ──────────────
# Sensibilidad y especificidad del algoritmo AliveCor para FA
# Fuente: Tung, N.L. et al. (2020). Evaluation of the AliveCor KardiaMobile
#   6L for detecting atrial fibrillation. JAMA Cardiology, 5(11), 1271–1277.
#   https://doi.org/10.1001/jamacardio.2020.4232
# (Tung et al. 2020, JAMA Cardiol.)

sens      = 0.97   # P(+|FA)  — sensibilidad
spec      = 0.96   # P(-|no FA) — especificidad
prev_FA   = 0.02   # prevalencia de FA en adultos >65 años en consulta general

# Regla de Bayes para VPP y VPN
p_pos_FA    = sens                     # P(+|FA)
p_pos_noFA  = 1 - spec                 # P(+|no FA) — tasa de falsos positivos
p_pos       = p_pos_FA * prev_FA + p_pos_noFA * (1 - prev_FA)  # P(+) — marginal

VPP = (p_pos_FA * prev_FA) / p_pos              # P(FA|+)
VPN = (spec * (1 - prev_FA)) / (1 - p_pos)     # P(no FA|-)

print('Prueba de FA (AliveCor) — Ejemplo 1')
print(f'  Sensibilidad : {sens:.0%}   Especificidad: {spec:.0%}')
print(f'  Prevalencia  : {prev_FA:.1%}')
print(f'  P(+)  (marginal) : {p_pos:.4f}')
print(f'  VPP = P(FA|+)    : {VPP:.3f}  ({VPP:.1%})')
print(f'  VPN = P(no FA|-) : {VPN:.3f}  ({VPN:.1%})')
print()
print('Interpretación: aunque el test es excelente (97%/96%),')
print(f'con una prevalencia del {prev_FA:.0%} solo {VPP:.0%} de los positivos')
print('realmente tienen FA — la falacia de la tasa base en acción.')

## Parte 2 — Par conjugado Beta-Binomial

Cuando el modelo es Bernoulli ($\mathcal{D}$ = número de éxitos en $N$ ensayos) y el prior es Beta,
el posterior es también Beta — esto se llama **conjugación**.

$$p(\theta) = \text{Beta}(\alpha, \beta) \qquad \Rightarrow \qquad p(\theta \mid \mathcal{D}) = \text{Beta}(\alpha + m,\; \beta + (N-m))$$

donde $m$ = número de éxitos, $N-m$ = número de fracasos.

**Interpretación:** $\alpha$ y $\beta$ actúan como *conteos pseudo-observados* previos.
Un prior $\text{Beta}(1,1)$ es uniforme (ignorancia completa).
Un prior $\text{Beta}(10,10)$ expresa que esperamos cerca del 50% de éxitos con moderada certeza.

### 2.1 Actualización secuencial — estimación de la sensibilidad de un biomarcador

In [ ]:
# ── Estimación bayesiana de la sensibilidad de un biomarcador de sepsis ────────
# Observamos resultados positivos/negativos en pacientes con sepsis confirmada
# y actualizamos nuestra creencia sobre la sensibilidad θ

theta_real = 0.82   # sensibilidad verdadera (desconocida en la práctica)
N_total    = 60     # pacientes con sepsis confirmada

# Simular resultados del biomarcador
resultados = rng.binomial(1, theta_real, N_total)  # 1=positivo, 0=negativo

# Tres priors distintos
priors = [
    (1,  1,  'Beta(1,1) — uniforme (sin info previa)',  'steelblue'),
    (5,  2,  'Beta(5,2) — optimista (θ≈0.7)',           'tomato'),
    (2,  8,  'Beta(2,8) — pesimista (θ≈0.2)',           'seagreen'),
]

# Puntos de actualización secuencial
checkpoints = [1, 5, 10, 20, 40, 60]
theta_grid  = np.linspace(0, 1, 500)

fig, ejes = plt.subplots(2, 3, figsize=(13, 7), sharex=True)
ejes_flat = ejes.flatten()

for k, n_obs in enumerate(checkpoints):
    ax = ejes_flat[k]
    m  = resultados[:n_obs].sum()   # éxitos observados hasta n_obs

    for a0, b0, etiqueta, color in priors:
        a_post = a0 + m
        b_post = b0 + (n_obs - m)
        posterior = stats.beta(a_post, b_post)
        ax.plot(theta_grid, posterior.pdf(theta_grid),
                lw=2, color=color,
                label=f'{etiqueta}\nMedia={posterior.mean():.2f}')

    ax.axvline(theta_real, color='k', ls='--', lw=1.2, label=f'θ real={theta_real}')
    ax.set(title=f'n = {n_obs}  (m={m} positivos)',
           xlabel='θ (sensibilidad)' if k >= 3 else '',
           ylabel='Densidad' if k % 3 == 0 else '')
    if k == 0:
        ax.legend(fontsize=7, loc='upper left')

fig.suptitle(
    'Actualización secuencial Beta-Binomial — estimación de la sensibilidad de un biomarcador\n'
    'Los tres priors convergen al mismo posterior al crecer n',
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()

print('Observación clave:')
print('  Con datos suficientes, todos los priors convergen al mismo posterior.')
print('  La verosimilitud "aplasta" el prior cuando n → ∞.')
print('  Con pocos datos, el prior importa mucho — elegirlo bien es crucial.')

## Parte 3 — Par conjugado Gaussiana-Gaussiana

Cuando los datos son gaussianos con varianza conocida $\sigma^2$
y el prior sobre la media es también gaussiano:

$$p(\mu) = \mathcal{N}(\mu_0, \sigma_0^2) \quad \Rightarrow \quad
p(\mu \mid \mathcal{D}) = \mathcal{N}(\mu_N, \sigma_N^2)$$

con:

$$\frac{1}{\sigma_N^2} = \frac{1}{\sigma_0^2} + \frac{N}{\sigma^2} \qquad
\mu_N = \sigma_N^2 \left(\frac{\mu_0}{\sigma_0^2} + \frac{N\bar{x}}{\sigma^2}\right)$$

La media posterior es un **promedio ponderado** entre la media prior $\mu_0$ y la media muestral $\bar{x}$,
con pesos proporcionales a las precisiones (inversas de varianza).

In [ ]:
# ── Gaussiana-Gaussiana: estimación del intervalo QT medio en una cohorte ──────
# QT corregido (QTc, ms) en pacientes con nuevo fármaco antiarrítmico
# Prior basado en valores de referencia de la literatura

mu_0    = 410.0   # ms — media prior (valores de referencia normales)
sigma_0 = 30.0    # ms — incertidumbre prior
sigma   = 20.0    # ms — varianza del proceso (conocida por datos históricos)
mu_real = 430.0   # ms — media real (el fármaco alarga el QTc)

ns_seq = [1, 2, 5, 10, 20, 50]
datos  = rng.normal(mu_real, sigma, max(ns_seq))

fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True)
x_grid = np.linspace(340, 510, 500)

for k, n in enumerate(ns_seq):
    ax = axes.flatten()[k]
    xbar = datos[:n].mean()

    # Actualización bayesiana gaussiana
    sigma_N_sq = 1 / (1/sigma_0**2 + n/sigma**2)
    mu_N       = sigma_N_sq * (mu_0/sigma_0**2 + n*xbar/sigma**2)
    sigma_N    = np.sqrt(sigma_N_sq)

    # Peso relativo del prior vs datos
    peso_prior = (1/sigma_0**2) / (1/sigma_0**2 + n/sigma**2)
    peso_datos = 1 - peso_prior

    ax.plot(x_grid, stats.norm(mu_0,  sigma_0 ).pdf(x_grid),
            'gray', lw=1.5, ls='--', label=f'Prior  μ={mu_0:.0f}')
    ax.plot(x_grid, stats.norm(mu_N,  sigma_N ).pdf(x_grid),
            'steelblue', lw=2.5, label=f'Post.  μ={mu_N:.1f}')
    ax.fill_between(x_grid, stats.norm(mu_N, sigma_N).pdf(x_grid),
                     alpha=0.15, color='steelblue')
    ax.axvline(mu_real, color='tomato', ls=':', lw=1.5, label=f'μ real={mu_real}')
    ax.axvline(xbar,    color='k',      ls=':',  lw=1,   label=f'x̄={xbar:.1f}')

    ax.set(title=f'n={n}  (x̄={xbar:.1f} ms)\n'
                  f'Prior:{peso_prior:.0%} / Datos:{peso_datos:.0%}',
           xlabel='QTc (ms)' if k >= 3 else '',
           ylabel='Densidad' if k % 3 == 0 else '')
    if k == 0:
        ax.legend(fontsize=7.5)

fig.suptitle(
    'Actualización bayesiana Gaussiana-Gaussiana — QTc bajo nuevo antiarrítmico\n'
    'La media posterior es un promedio ponderado entre el prior y los datos',
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()

## Parte 4 — MAP vs EMV: la conexión con la regularización

La estimación MAP (máximo a posteriori) busca el modo de la distribución posterior:

$$\hat{\theta}_{\text{MAP}} = \arg\max_\theta \left[\log p(\mathcal{D}\mid\theta) + \log p(\theta)\right]$$

Con un prior gaussiano $p(\theta) = \mathcal{N}(0, \tau^2)$ sobre los pesos, el término de log-prior es:

$$\log p(\theta) = -\frac{\|\theta\|^2}{2\tau^2} + \text{cte}$$

Esto es exactamente la **regularización L2** (ridge) con $\lambda = 1/\tau^2$.
Un prior más estrecho ($\tau$ pequeño) ↔ regularización más fuerte.

In [ ]:
# ── MAP vs EMV en regresión lineal — efecto del prior gaussiano ────────────────
# Predecir la dosis de insulina (UI) a partir del nivel de glucosa (mg/dL)

N_pts = 15
glucosa = rng.uniform(80, 300, N_pts)
dosis   = 0.05 * glucosa - 2.0 + rng.normal(0, 1.5, N_pts)

# Diseño con intercepto
X = np.column_stack([np.ones(N_pts), glucosa])
y = dosis

# EMV (mínimos cuadrados ordinarios)
theta_emv = np.linalg.lstsq(X, y, rcond=None)[0]

# MAP con distintos priors gaussianos (equivalente a ridge con λ=sigma²/tau²)
sigma2 = 1.5**2   # varianza del proceso
taus   = [0.5, 2.0, 10.0]   # desviaciones estándar del prior (estrecho → amplio)

g_plot = np.linspace(70, 310, 200)
X_plot = np.column_stack([np.ones(200), g_plot])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo: curvas de regresión
axes[0].scatter(glucosa, dosis, s=50, color='steelblue', alpha=0.7, label='Datos')
axes[0].plot(g_plot, X_plot @ theta_emv, 'k-', lw=2.5, label=f'EMV (sin prior)')

colores_tau = ['tomato', 'darkorange', 'seagreen']
for tau, color in zip(taus, colores_tau):
    lam = sigma2 / tau**2
    # Solución MAP: (XᵀX + λI)⁻¹ Xᵀy
    theta_map = np.linalg.solve(X.T @ X + lam * np.eye(2), X.T @ y)
    axes[0].plot(g_plot, X_plot @ theta_map, '--', lw=1.8, color=color,
                  label=f'MAP τ={tau} (λ={lam:.2f})')

axes[0].set(xlabel='Glucosa (mg/dL)', ylabel='Dosis insulina (UI)',
            title='MAP vs EMV — prior gaussiano = regularización L2\n'
                  'Prior estrecho (τ pequeño) = más regularización')
axes[0].legend(fontsize=8)

# Panel derecho: magnitud de los coeficientes vs λ (trayectoria ridge)
lambdas = np.logspace(-2, 2, 100)
coefs   = []
for lam in lambdas:
    th = np.linalg.solve(X.T @ X + lam * np.eye(2), X.T @ y)
    coefs.append(th)
coefs = np.array(coefs)

axes[1].semilogx(lambdas, coefs[:, 0], 'steelblue', lw=2, label='Intercepto (β₀)')
axes[1].semilogx(lambdas, coefs[:, 1], 'tomato',    lw=2, label='Pendiente (β₁)')
axes[1].axhline(theta_emv[0], color='steelblue', ls=':', lw=1)
axes[1].axhline(theta_emv[1], color='tomato',    ls=':', lw=1)
axes[1].axhline(0, color='gray', lw=0.8)
axes[1].set(xlabel='λ = σ²/τ²  (regularización)', ylabel='Valor del coeficiente',
            title='Trayectoria ridge\nλ→0: EMV  |  λ→∞: coeficientes → 0')
axes[1].legend(fontsize=9)
for tau in taus:
    axes[1].axvline(sigma2/tau**2, color='gray', ls='--', lw=0.8, alpha=0.5)

plt.tight_layout()
plt.show()

print('Equivalencias clave:')
print('  Prior gaussiano estrecho (τ pequeño) ↔ λ grande ↔ coeficientes más pequeños')
print('  Prior gaussiano amplio   (τ grande)  ↔ λ pequeño ↔ solución cercana a EMV')
print('  Prior uniforme (τ→∞)                ↔ λ=0 ↔ EMV exacto')

## Parte 5 — La falacia de la tasa base: VPP, VPN y prevalencia

**VPP** (Valor Predictivo Positivo) = $P(\text{enfermedad} \mid \text{prueba}^+)$

**VPN** (Valor Predictivo Negativo) = $P(\text{sano} \mid \text{prueba}^-)$

Ambos dependen críticamente de la **prevalencia** $p$ — aunque la sensibilidad y especificidad no cambien.
Esto es la **falacia de la tasa base**: intuir que un test excelente implica un VPP alto,
ignorando que con prevalencia baja la mayoría de positivos son falsos.

> **Experimento clásico (Casscells et al., NEJM 1978):**
> Se preguntó a médicos del Harvard Medical School:
> *"Si una prueba tiene 5% de falsos positivos y la prevalencia es 1/1000,
> ¿cuál es la probabilidad de enfermedad dado un resultado positivo?"*
> La respuesta correcta es ~2%, pero la mayoría respondió 95%.

In [ ]:
# ── VPP y VPN en función de la prevalencia ────────────────────────────────────

prevalencias = np.linspace(0.001, 0.50, 500)

# Tres escenarios de prueba diagnóstica
pruebas = [
    (0.97, 0.96, 'AliveCor FA\n(Sens=97%, Espec=96%)',     'steelblue'),
    (0.85, 0.90, 'PCR sepsis\n(Sens=85%, Espec=90%)',       'tomato'),
    (0.99, 0.99, 'Prueba ideal\n(Sens=99%, Espec=99%)',     'seagreen'),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for sens, spec, etiqueta, color in pruebas:
    p_pos = sens * prevalencias + (1-spec) * (1-prevalencias)
    vpp   = sens * prevalencias / p_pos
    p_neg = (1-sens) * prevalencias + spec * (1-prevalencias)
    vpn   = spec * (1-prevalencias) / p_neg

    axes[0].plot(prevalencias * 100, vpp * 100, lw=2.5, color=color, label=etiqueta)
    axes[1].plot(prevalencias * 100, vpn * 100, lw=2.5, color=color, label=etiqueta)

# Marcar prevalencias clínicas relevantes
marcas = [
    (0.2,  'FA en >65a\nconsulta general'),
    (2.0,  'FA en\npoblación general'),
    (10.0, 'Sepsis en\nUCI'),
]
for prev_pct, etiqueta in marcas:
    for ax in axes:
        ax.axvline(prev_pct, color='gray', ls=':', lw=1, alpha=0.7)
        ax.text(prev_pct, 5, etiqueta, fontsize=7, color='gray',
                ha='center', va='bottom', rotation=90)

axes[0].set(xlabel='Prevalencia (%)', ylabel='VPP (%)',
            title='Valor Predictivo Positivo vs Prevalencia\n'
                  'El VPP colapsa a bajas prevalencias')
axes[0].legend(fontsize=8)
axes[1].set(xlabel='Prevalencia (%)', ylabel='VPN (%)',
            title='Valor Predictivo Negativo vs Prevalencia\n'
                  'El VPN se mantiene alto con prevalencia baja')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print('Mensaje clave de la falacia de la tasa base:')
print('─' * 55)
for sens, spec, etiqueta, _ in pruebas:
    for prev in [0.002, 0.02, 0.10]:
        p_pos = sens*prev + (1-spec)*(1-prev)
        vpp   = sens*prev / p_pos
        print(f'  {etiqueta.split(chr(10))[0]:20s} '
              f'prev={prev:.1%}  VPP={vpp:.1%}')

## Parte 6 — Tabla de contingencia 2×2 y cociente de verosimilitud

La visión frecuentista complementa la bayesiana mediante la **tabla de contingencia** y
el **cociente de verosimilitud** (likelihood ratio, LR), que es independiente de la prevalencia:

$$LR^+ = \frac{\text{Sensibilidad}}{1 - \text{Especificidad}} = \frac{P(+\mid E)}{P(+\mid \bar{E})}$$

$$\text{Odds post-prueba} = LR^+ \times \text{Odds pre-prueba}$$

In [ ]:
# ── Tabla de contingencia interactiva ─────────────────────────────────────────
def tabla_contingencia(sens, spec, prevalencia, N=10000):
    """
    Construye la tabla 2×2 para N pacientes hipotéticos.
    Retorna dict con todas las métricas de rendimiento diagnóstico.
    """
    enfermos = round(N * prevalencia)
    sanos    = N - enfermos

    VP = round(enfermos * sens)          # verdaderos positivos
    FN = enfermos - VP                   # falsos negativos
    FP = round(sanos * (1 - spec))       # falsos positivos
    VN = sanos - FP                      # verdaderos negativos

    VPP_t = VP / (VP + FP) if (VP + FP) > 0 else 0
    VPN_t = VN / (VN + FN) if (VN + FN) > 0 else 0
    LRp   = sens / (1 - spec + 1e-9)
    LRn   = (1 - sens) / (spec + 1e-9)

    return {'VP': VP, 'FP': FP, 'FN': FN, 'VN': VN,
            'VPP': VPP_t, 'VPN': VPN_t, 'LR+': LRp, 'LR-': LRn,
            'Exactitud': (VP+VN)/N}


# Comparar tres pruebas de sepsis a prevalencia del 10% (UCI)
prev_uci = 0.10
escenarios = [
    (0.85, 0.90, 'Procalcitonina\n(PCT > 2 ng/mL)'),
    (0.70, 0.95, 'Lactato sérico\n(> 2 mmol/L)'),
    (0.92, 0.85, 'PCR > 100 mg/L'),
]

print(f'Comparación de biomarcadores de sepsis (prevalencia UCI = {prev_uci:.0%})')
print(f'{"Biomarcador":<25} {"VP":>5} {"FP":>5} {"FN":>5} {"VN":>5}'
      f' {"VPP":>6} {"VPN":>6} {"LR+":>6} {"Exactitud":>10}')
print('─' * 80)

for sens, spec, etiqueta in escenarios:
    r = tabla_contingencia(sens, spec, prev_uci)
    nombre = etiqueta.replace('\n', ' ')
    print(f'{nombre:<25} {r["VP"]:>5} {r["FP"]:>5} {r["FN"]:>5} {r["VN"]:>5}'
          f' {r["VPP"]:>6.1%} {r["VPN"]:>6.1%} {r["LR+"]:>6.1f} {r["Exactitud"]:>10.1%}')

print()
print('LR+ > 10 → cambio diagnóstico muy significativo')
print('LR+ 5–10 → cambio moderado')
print('LR+ < 2  → cambio mínimo — prueba poco útil')

# Visualización de la tabla 2×2 para PCT
sens_pct, spec_pct = 0.85, 0.90
r = tabla_contingencia(sens_pct, spec_pct, prev_uci)

fig, ax = plt.subplots(figsize=(6, 4))
ax.set_xlim(0, 2); ax.set_ylim(0, 2)
ax.axis('off')

datos_tabla = [
    ['',           'Sepsis (+)',  'Sano (-)'],
    ['Test (+)',   str(r['VP']),  str(r['FP'])],
    ['Test (-)',   str(r['FN']),  str(r['VN'])],
]
colores_tabla = [
    ['#1B3A6B', '#1B3A6B',   '#1B3A6B'],
    ['#1B3A6B', '#DCFCE7',   '#FEE2E2'],
    ['#1B3A6B', '#FEE2E2',   '#DCFCE7'],
]
for i, fila in enumerate(datos_tabla):
    for j, val in enumerate(fila):
        bg = colores_tabla[i][j]
        fc = 'white' if bg == '#1B3A6B' else '#1B3A6B'
        ax.add_patch(plt.Rectangle((j*0.67, 1.6-i*0.75), 0.65, 0.70,
                                     color=bg, zorder=2))
        weight = 'bold' if i == 0 or j == 0 else 'normal'
        ax.text(j*0.67 + 0.325, 1.95 - i*0.75, val,
                ha='center', va='center', fontsize=13,
                color=fc, fontweight=weight, zorder=3)

ax.set_title(f'Tabla de contingencia — PCT para sepsis\n'
              f'(N=10,000  |  Prevalencia={prev_uci:.0%}  |  '
              f'VPP={r["VPP"]:.1%}  VPN={r["VPN"]:.1%})',
              fontsize=11)
plt.tight_layout()
plt.show()

## Parte 7 — Análisis de sensibilidad del prior

Una pregunta práctica importante: **¿cuánto importa la elección del prior?**
Esta celda permite explorar sistemáticamente cómo distintos priors afectan el posterior.

In [ ]:
# ── Sensibilidad al prior — ¿cuántos datos necesito para que el prior no importe? ──
# Modelamos la tasa de detección de apneas en una PSG (polisomnografía)

theta_real_apnea = 0.65   # tasa real de detección
N_psg = 80                # pacientes en el estudio
m_obs = round(theta_real_apnea * N_psg)

# Distintos priors Beta
priors_sens = [
    (1,  1,  'Uniforme Beta(1,1)'),
    (10, 2,  'Optimista Beta(10,2) — θ≈0.83'),
    (2,  10, 'Pesimista Beta(2,10) — θ≈0.17'),
    (5,  5,  'Centrado Beta(5,5) — θ≈0.50'),
    (30, 20, 'Informativo Beta(30,20) — θ≈0.60'),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
theta_g   = np.linspace(0, 1, 500)
colores_s = ['steelblue', 'tomato', 'seagreen', 'darkorange', 'mediumpurple']

for (a0, b0, lbl), color in zip(priors_sens, colores_s):
    # Prior
    axes[0].plot(theta_g, stats.beta(a0, b0).pdf(theta_g),
                  lw=1.5, ls='--', color=color, alpha=0.6)
    # Posterior
    a_p, b_p = a0 + m_obs, b0 + (N_psg - m_obs)
    post_pdf  = stats.beta(a_p, b_p).pdf(theta_g)
    axes[1].plot(theta_g, post_pdf, lw=2.5, color=color,
                  label=f'{lbl}\n  → Post. media={a_p/(a_p+b_p):.3f}')

for ax in axes:
    ax.axvline(theta_real_apnea, color='k', ls=':', lw=1.5, label='θ real=0.65')
    ax.axvline(m_obs/N_psg,      color='gray', ls='-.', lw=1,
                label=f'EMV={m_obs/N_psg:.2f}')

axes[0].set(title=f'Priors (líneas discontinuas)\n'
                   f'N={N_psg}, m={m_obs} detecciones observadas',
            xlabel='θ (tasa de detección)', ylabel='Densidad')
axes[1].set(title='Posteriors — todos convergen\n'
                   'con N=80 el prior ya no importa demasiado',
            xlabel='θ (tasa de detección)')
axes[1].legend(fontsize=7.5, loc='upper left')

plt.suptitle('Análisis de sensibilidad del prior — tasa de detección de apneas (PSG)',
              fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print(f'Con N={N_psg} observaciones, la media posterior varía entre:')
medias = [(a0+m_obs)/(a0+b0+N_psg) for a0, b0, _ in priors_sens]
print(f'  Mínimo: {min(medias):.3f}   Máximo: {max(medias):.3f}')
print(f'  Rango: {max(medias)-min(medias):.3f}  (comparar con θ_EMV={m_obs/N_psg:.3f})')

## ✏️ Ejercicios

1. **VPP para tamizaje neonatal.** El tamizaje de hipotiroidismo congénito en México (IMSS/SSA)
   tiene sensibilidad del 98.7% y especificidad del 99.5%. La prevalencia es 1 en 3,000 nacidos vivos.
   (a) Calcula el VPP. ¿Cuántos de cada 100 positivos realmente tienen hipotiroidismo?  
   (b) ¿Qué especificidad se necesitaría para lograr VPP ≥ 50%?  
   (c) Grafica VPP vs especificidad para prevalencias de 1/500, 1/1000 y 1/3000.

2. **Actualización secuencial Beta-Binomial.** Repite el experimento de la Parte 2 pero
   procesa las observaciones de una en una, graficando cómo evoluciona la media y la varianza
   posterior con cada nuevo paciente. ¿Cuántos pacientes se necesitan para que el intervalo
   de credibilidad del 95% sea más estrecho que ±0.05?

3. **Comparación MAP vs EMV con datos escasos.** Con solo N=5 pacientes, ajusta un modelo
   de regresión (glucosa → dosis insulina) usando EMV y MAP con tres priors distintos.
   Evalúa el error de predicción en 50 nuevos pacientes simulados. ¿Cuál prior produce
   menor error de generalización? ¿Por qué?

4. **Cociente de verosimilitud en cascada.** En la práctica clínica se aplican múltiples
   pruebas secuencialmente. Si aplicas primero PCR (LR+=8.7) y luego procalcitonina
   (LR+=9.5) asumiendo independencia condicional, ¿cuánto aumentan los odds de sepsis
   si ambas son positivas? Partiendo de una prevalencia del 5%, calcula la probabilidad
   post-prueba combinada.

5. *(Desafío)* **Prior conjugado para la distribución de Poisson.** La distribución de Poisson
   $p(k\mid\lambda) = e^{-\lambda}\lambda^k/k!$ modela conteos de eventos (p. ej., crisis
   epilépticas por semana). El prior conjugado es la distribución Gamma.
   (a) Deriva la actualización posterior Gamma a partir del prior Gamma(α,β).
   (b) Implementa la actualización secuencial para 20 semanas de observación de un paciente
   con λ_real = 2.3 crisis/semana.

## 📚 Conjuntos de datos utilizados / referenciados

| Conjunto de datos | Fuente | Notas |
|---|---|---|
| AliveCor FA (sens/espec) | Tung, N.L. et al. (2020). *JAMA Cardiol.* 5(11):1271–1277. https://doi.org/10.1001/jamacardio.2020.4232 | Sens=97%, Espec=96% para detección de FA |
| Tamizaje neonatal hipotiroidismo | INSP / SSA México | Prevalencia 1:3,000 nacidos vivos |
| Biomarcadores sepsis | Rhodes et al. (2016). *JAMA* 315(8):801–810 (Sepsis-3) | PCT, lactato, PCR |
| QTc antiarrítmico | Datos simulados | Calibrado con valores de referencia AHA/ESC |